# Plots maps fLUC differences

In [ ]:
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
# Directories
dir_nearth = '../data/naturalearth/'
dir_zen ='../paper_ecosystem_carbon/output/zenodo/'
dir08 = '../paper_ecosystem_carbon/output/08_eval/'

---

### Pre-processing

In [ ]:
# Get cspatial carbon density data
ds_in = xr.open_dataset(f'{dir_zen}ecosystem_carbon_data_figure_m05.nc') 

# Get coastline data
# Download: https://www.naturalearthdata.com/
coastline110 = gpd.read_file(
    f'{dir_nearth}ne_110m_coastline/ne_110m_coastline.shp')

In [ ]:
# Land-sea mask
da_land = ds_in.chansis_fluc_net[0].notnull()

# Split and sum data
def get_ds_subsum(s):
    return (
        ds_in[[var for var in ds_in.data_vars if var.startswith(f'{s}_')]]
            .sum(['time'])
            .rename({
                var: var.removeprefix(f'{s}_') 
                for var in ds_in.data_vars
                if var.startswith(f'{s}_')
            })
    )

ds_chansis = get_ds_subsum('chansis')
ds_cspatial = get_ds_subsum('cspatial')
ds_cdgvm = get_ds_subsum('cdgvm')

# Prepare dictionaries for plotting
dict_ds = {
    'chansis': (ds_chansis, 'BLUE cHansis'), 
    'cspatial': (ds_cspatial, 'BLUE cSpatial'), 
    'cdgvm': (ds_cdgvm, 'BLUE cDGVM'),
    'cspatial-chansis': 
    ((ds_cspatial - ds_chansis), 'BLUE cSpatial - BLUE cHansis'),
    'cspatial-cdgvm': 
    ((ds_cspatial - ds_cdgvm), 'BLUE cSpatial - BLUE cDGVM'),
    'cdgvm-chansis':  
    ((ds_cdgvm - ds_chansis), 'BLUE cDGVM - BLUE cHansis')   
}

dict_vars = {
    'net': 'Net fluxes', 
    'deforestation': 'Deforestation', 
    'forest_regrowth': 'Forest (re-)growth',
    'wood_harvest': 'Wood harvest', 
    'other_transitions': 'Other transitions'
}

---

### Plot functions

In [ ]:
def plot_sub_maps(ax, what, cmap, vmin, vmax):
    
    """Returns fluc or fluc difference subplot"""
    
    s, var = what.split('_', 1)
    
    # Subplot
    ds, label = dict_ds[s]
    da = ds[f'fluc_{var}'].where(da_land)

    im = da.plot.imshow(
        ax=ax, vmin=vmin, vmax=vmax, cmap=cmap, add_colorbar=False
    )

    coastline110.plot(ax=ax, color='#000000', linewidth=0.5)

    ax.set_title(f'{dict_vars[var]} ({label})')
    ax.axis('off')
    ax.set_ylim(-60, 85)
    
    return im


def plot_sub_add_colorbar(ax, im, extend, pad=0.05, aspect=35, shrink=0.8):
    
    """ Plots colorbar"""    
    
    cbar = fig.colorbar(im, ax=ax,
                        orientation='horizontal', pad=pad, 
                        label=r'Cumulative CO$_{2}$ flux 1960–2023 (tC ha$^{-1}$)', 
                        aspect=aspect, shrink=shrink, extend=extend)
    return cbar

---

### Plot figure s06

In [ ]:
# Define min and max values
v0 = 70
v1 = 20

# Plot
fig, axs = plt.subplots(figsize=(12, 14), ncols=2, nrows=5, 
                        constrained_layout=True, dpi=300)

# Subplots left
im00 = plot_sub_maps(axs[0,0], 'cspatial_net', 'PiYG_r', -v0, v0)
im10 = plot_sub_maps(axs[1,0], 'cspatial_deforestation', 'PiYG_r', -v0, v0)
im20 = plot_sub_maps(axs[2,0], 'cspatial_forest_regrowth', 'PiYG_r', -v0, v0)
im30 = plot_sub_maps(axs[3,0], 'cspatial_wood_harvest', 'PiYG_r', -v0, v0)
im40 = plot_sub_maps(axs[4,0], 'cspatial_other_transitions', 'PiYG_r', -v0, v0)

# Legend left
cbar_left = plot_sub_add_colorbar(axs[4,0], im00, 'both')
cbar_left_ax = cbar_left.ax
cbar_left_ax.text(-v0, 1.8, f' ', ha='left', va='center', c='#000000')
cbar_left_ax.text(v0, 1.8, f' ', ha='right', va='center', c='#000000');

# Subplots right
im01 = plot_sub_maps(axs[0,1], 'cspatial-chansis_net', 'RdBu_r', -v1, v1)
im11 = plot_sub_maps(axs[1,1], 'cspatial-chansis_deforestation', 'RdBu_r', -v1, v1)
im21 = plot_sub_maps(axs[2,1], 'cspatial-chansis_forest_regrowth', 'RdBu_r', -v1, v1)
im31 = plot_sub_maps(axs[3,1], 'cspatial-chansis_wood_harvest', 'RdBu_r', -v1, v1)
im41 = plot_sub_maps(axs[4,1], 'cspatial-chansis_other_transitions', 'RdBu_r', -v1, v1)

# Legend right
cbar_right = plot_sub_add_colorbar(axs[4,1], im01, 'both')
cbar_right_ax = cbar_right.ax
cbar_right_ax.text(-v1, 1.8, 'BLUE cHansis higher', ha='left', va='center', c='#000000')
cbar_right_ax.text(v1, 1.8, 'BLUE cSpatial higher', ha='right', va='center', c='#000000')

# Export
fstr_out = 'fig_s06_fluc_maps1'
plt.savefig(f'{dir08}pdf/{fstr_out}.pdf', dpi=300, 
            bbox_inches='tight', pad_inches=0.1); 
plt.savefig(f'{dir08}png/{fstr_out}.png', dpi=300, 
            bbox_inches='tight', pad_inches=0.1);

---

### Plot figure f07

In [ ]:
# Define min and max values
v0 = 20
v1 = 20

# Plot
fig, axs = plt.subplots(figsize=(12, 14), ncols=2, nrows=5, 
                        constrained_layout=True, dpi=300)

# Subplots left
im00 = plot_sub_maps(axs[0,0], 'cspatial-cdgvm_net', 'RdBu_r', -v0, v0)
im10 = plot_sub_maps(axs[1,0], 'cspatial-cdgvm_deforestation', 'RdBu_r', -v0, v0)
im20 = plot_sub_maps(axs[2,0], 'cspatial-cdgvm_forest_regrowth', 'RdBu_r', -v0, v0)
im30 = plot_sub_maps(axs[3,0], 'cspatial-cdgvm_wood_harvest', 'RdBu_r', -v0, v0)
im40 = plot_sub_maps(axs[4,0], 'cspatial-cdgvm_other_transitions', 'RdBu_r', -v0, v0)

# Legend left
cbar_left = plot_sub_add_colorbar(axs[4,0], im40, 'both')
cbar_left_ax = cbar_left.ax
cbar_left_ax.text(-v0, 1.8, 'BLUE cDGVM higher', ha='left', va='center', c='#000000')
cbar_left_ax.text(v0, 1.8, 'BLUE cSpatial higher', ha='right', va='center', c='#000000')

# Subplots right
im01 = plot_sub_maps(axs[0,1], 'cdgvm-chansis_net', 'RdBu_r', -v1, v1)
im11 = plot_sub_maps(axs[1,1], 'cdgvm-chansis_deforestation', 'RdBu_r', -v1, v1)
im21 = plot_sub_maps(axs[2,1], 'cdgvm-chansis_forest_regrowth', 'RdBu_r', -v1, v1)
im31 = plot_sub_maps(axs[3,1], 'cdgvm-chansis_wood_harvest', 'RdBu_r', -v1, v1)
im41 = plot_sub_maps(axs[4,1], 'cdgvm-chansis_other_transitions', 'RdBu_r', -v1, v1)

# Legend right
cbar_right = plot_sub_add_colorbar(axs[4,1], im41, 'both')
cbar_right_ax = cbar_right.ax
cbar_right_ax.text(-v1, 1.8, 'BLUE cHansis higher', ha='left', va='center', c='#000000')
cbar_right_ax.text(v1, 1.8, 'BLUE cDGVM higher', ha='right', va='center', c='#000000')

# Export
fstr_out = 'fig_s07_fluc_maps2'
plt.savefig(f'{dir08}pdf/{fstr_out}.pdf', dpi=300, 
            bbox_inches='tight', pad_inches=0.1); 
plt.savefig(f'{dir08}png/{fstr_out}.png', dpi=300, 
            bbox_inches='tight', pad_inches=0.1);